# Silver to Gold - Legacy

In [ ]:
# configuracion
k = dbutils.secrets.get(scope="data-demo-scope", key="storage-account-key")
sa = "stadatademo2026"
spark.conf.set(f"fs.azure.account.key.{sa}.dfs.core.windows.net", k)
jdbc = dbutils.secrets.get(scope="data-demo-scope", key="sql-connection-string")

In [ ]:
# leer datos silver
df1 = spark.read.format("delta").load(f"abfss://silver@{sa}.dfs.core.windows.net/autores")
df2 = spark.read.format("delta").load(f"abfss://silver@{sa}.dfs.core.windows.net/libros")
df3 = spark.read.format("delta").load(f"abfss://silver@{sa}.dfs.core.windows.net/usuarios")
df4 = spark.read.format("delta").load(f"abfss://silver@{sa}.dfs.core.windows.net/prestamos")

In [ ]:
# escribir autores a sql
df1.write.format("jdbc").option("url", jdbc).option("dbtable", "autores").option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver").mode("overwrite").save()
print("ok autores")

In [ ]:
# escribir libros a sql
df2.write.format("jdbc").option("url", jdbc).option("dbtable", "libros").option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver").mode("overwrite").save()
print("ok libros")

In [ ]:
# escribir usuarios a sql  
df3.write.format("jdbc").option("url", jdbc).option("dbtable", "usuarios").option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver").mode("overwrite").save()
print("ok usuarios")

In [ ]:
# escribir prestamos a sql
df4.write.format("jdbc").option("url", jdbc).option("dbtable", "prestamos").option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver").mode("overwrite").save()
print("ok prestamos")

In [ ]:
from pyspark.sql.functions import *

# dashboard - todo junto
res = df4.join(df2, df4.libro_id == df2.id, "left").join(df1, df2.autor_id == df1.id, "left").join(df3, df4.usuario_id == df3.id, "left").select(
    df4.id.alias("prestamo_id"),
    df2.titulo,
    df2.genero,
    df2.isbn,
    df1.nombre.alias("autor"),
    df1.nacionalidad,
    df3.nombre.alias("usuario"),
    df3.tipo_socio,
    df3.email,
    df4.fecha_prestamo,
    df4.fecha_devolucion,
    df4.estado,
    datediff(df4.fecha_devolucion, df4.fecha_prestamo).alias("dias")
)

In [ ]:
# calcular cosas extra
from pyspark.sql.functions import when, lit, current_date

res2 = res.withColumn("vencido", when(res.estado == "vencido", lit(True)).otherwise(lit(False)))
res3 = res2.withColumn("dias_retraso", when(res2.estado == "vencido", datediff(current_date(), res2.fecha_prestamo) - 14).otherwise(lit(0)))
res4 = res3.withColumn("mes_prestamo", month(res3.fecha_prestamo))
res5 = res4.withColumn("anio_prestamo", year(res4.fecha_prestamo))
res6 = res5.withColumn("trimestre", quarter(res5.fecha_prestamo))

In [ ]:
# guardar dashboard en sql
res6.write.format("jdbc").option("url", jdbc).option("dbtable", "prestamos_dashboard").option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver").mode("overwrite").save()
print("todo ok")